# Phase 8 - Encoder generalization (GATv2)

Does the framework's decision survive a change of aggregator? GraphSAGE stays the study's encoder; nothing here re-fits a frozen rule.

Only the convolution varies: `SAGEConv(mean)` -> `GATv2Conv(4 heads)`. Graph, role graph, features, split, objective, epochs and eval are identical.

Link prediction, K=10, seeds 42-44, seven variants, tested on total 30 graphs (`cfg.ENCODER_PANEL`).


In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from experiments import encoder_transfer as et

STYLE = [{"selector": "table", "props": [("font-size", "12px")]},
         {"selector": "th", "props": [("text-align", "center")]},
         {"selector": "td", "props": [("text-align", "center")]}]
show = lambda df: display(df.style.set_table_styles(STYLE).hide(axis="index"))

fourcell = pd.read_csv("results/encoder_transfer_fourcell.csv")
agree = pd.read_csv("results/encoder_transfer_agreement.csv")
perseed = pd.read_csv("results/encoder_transfer_perseed.csv")
PANEL = sorted(agree.dataset)
cross = et.cross(perseed, "gatv2_edge", "graphsage_edge").set_index("dataset")
gv = fourcell[fourcell.encoder == "gatv2_edge"].set_index("dataset")
w = fourcell.pivot(index="dataset", columns="encoder", values=["original", "best_aug", "gap", "aug_wins"])
w.columns = [f"{b.replace('_edge','')}_{a}" for a, b in w.columns]
print(f"{len(PANEL)} graphs")

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


30 graphs


## Q1 · Does augmentation still help when the encoder changes?

The four cells: `GraphSAGE`, `GraphSAGE+aug`, `GATv2`, `GATv2+aug`. WHERE `+aug` = the best non-`original` variant per graph and encoder.


In [2]:
tbl = pd.DataFrame({"original": [w.graphsage_original.mean(), w.gatv2_original.mean()],
                    "+aug":     [w.graphsage_best_aug.mean(), w.gatv2_best_aug.mean()]},
                   index=["GraphSAGE", "GATv2"]).round(4)
tbl["aug - original"] = (tbl["+aug"] - tbl["original"]).round(4)
tbl["aug wins"] = [f"{int(w.graphsage_aug_wins.sum())}/{len(w)}", f"{int(w.gatv2_aug_wins.sum())}/{len(w)}"]
print(f"MEAN AUC, {len(w)} graphs")
display(tbl)

best = w[["graphsage_original", "graphsage_best_aug", "gatv2_original", "gatv2_best_aug"]].idxmax(axis=1).value_counts()
print(f"\nBEST OF THE FOUR, counted over {len(w)} graphs")
display(pd.DataFrame([[best.get("graphsage_original", 0), best.get("graphsage_best_aug", 0)],
                      [best.get("gatv2_original", 0), best.get("gatv2_best_aug", 0)]],
                     index=["GraphSAGE", "GATv2"], columns=["original", "+aug"]))

MEAN AUC, 30 graphs


,original,+aug,aug - original,aug wins
GraphSAGE,0.6250,0.6712,0.0462,21/30
GATv2,0.6684,0.7170,0.0486,13/30



BEST OF THE FOUR, counted over 30 graphs


,original,+aug
GraphSAGE,4,5
GATv2,9,12


**A1.** Augmentation helps both encoders on average, but on far fewer graphs under GATv2 - **13/30 vs 21/30**.


## Q2 · Can GATv2 alone already beat GraphSAGE+aug?

GATv2 on the **original** graph vs GraphSAGE on its **best augmented** graph, paired on the shared split. A strict beat - a tie is not a win.


In [3]:
band = cross.paired_ratio.apply(lambda r: "GATv2 alone beats" if r > 1 else
                                ("no difference" if abs(r) <= 1 else "GATv2 alone loses"))
rows = []
for k in ["GATv2 alone beats", "no difference", "GATv2 alone loses"]:
    n = int((band == k).sum())
    helps = int(gv.loc[sorted(band[band == k].index)].aug_wins.sum())
    rows.append({"GATv2 alone vs GraphSAGE+aug": k, "graphs": f"{n} / {len(PANEL)}",
                 "of those, augmentation helps GATv2": f"{helps} / {n}"})
show(pd.DataFrame(rows))

GATv2 alone vs GraphSAGE+aug,graphs,"of those, augmentation helps GATv2"
GATv2 alone beats,14 / 30,2 / 14
no difference,7 / 30,3 / 7
GATv2 alone loses,9 / 30,8 / 9


**A2.** Yes on 14/30 - so the objection holds about half the time. But the two outcomes are near-disjoint: where GATv2 alone already wins, augmentation adds almost nothing (2 of 14); where GATv2 alone loses, augmentation is what rescues it (8 of 9).

**A stronger encoder (GATv2) sometimes removes the need for augmentation. but provides substantial gains when stronger encoder (GATv2) alone underperforms.**


## Q3 · On the graphs where GATv2 alone fails, can augmentation rescue it?

The 9 graphs from Q2 where GATv2 on the original graph loses to GraphSAGE+aug.


In [4]:
fail = sorted(band[band == "GATv2 alone loses"].index)
g = gv.loc[fail]
show(pd.DataFrame([
    {"where GATv2 alone fails": "graphs",                         "value": f"{len(g)} / {len(PANEL)}"},
    {"where GATv2 alone fails": "GATv2 alone, mean AUC",           "value": f"{g.original.mean():.4f}"},
    {"where GATv2 alone fails": "GATv2 + augmentation, mean AUC",  "value": f"{g.best_aug.mean():.4f}"},
    {"where GATv2 alone fails": "mean gain",                       "value": f"{(g.best_aug - g.original).mean():+.4f}"},
    {"where GATv2 alone fails": "graphs rescued",                  "value": f"{int(g.aug_wins.sum())} / {len(g)}"}]))

where GATv2 alone fails,value
graphs,9 / 30
"GATv2 alone, mean AUC",0.5072
"GATv2 + augmentation, mean AUC",0.6619
mean gain,+0.1547
graphs rescued,8 / 9


**A3.** Yes - augmentation rescues 8 of the 9 cases where GATv2 performs worse, lifting mean AUC from 0.5072 to 0.6619.


## Q4 · Do the framework's calls survive?

Stage 2 is conditional on stage 1 saying _augment_, so it is only scorable where both encoders augment and both resolve a single signal.

Question 4 checks whether the same “keep original vs augment, and which signal to use” decisions still stay mostly the same when we replace GraphSAGE with GATv2.


In [5]:
dis = agree[~agree.stage1_agree]
c = agree[agree.stage2_comparable]
both = c[(c.stage1_baseline == "augment") & (c.stage1_encoder == "augment")]
show(pd.DataFrame([
    {"call": "stage 1 - keep or augment", "GraphSAGE and GATv2 agree": f"{int(agree.stage1_agree.sum())} / {len(agree)}",
     "note": f"all {len(dis)} disagreements run augment -> keep; none the other way"},
    {"call": "stage 2 - which signal", "GraphSAGE and GATv2 agree": f"{int(both.stage2_agree.sum())} / {len(both)}",
     "note": "scored only where both encoders augment"}]))

call,GraphSAGE and GATv2 agree,note
stage 1 - keep or augment,22 / 30,all 8 disagreements run augment -> keep; none the other way
stage 2 - which signal,5 / 7,scored only where both encoders augment


**A4.** Stage 1 survives 22/30, and every failure is one-directional - GATv2 makes augmentation unnecessary which means its already stronger without augmentation on some spefific cases not all. Stage 2 survives 5/7 (we tested only the 7 cases which passed stage 1 filter of augmentation verdict).

Q4 supports the same idea: when we switched to stronger GATv2, 8 graphs changed from “augment” to “keep original,” while augmentation still helped 8/9 graphs where GATv2 was weak.


## Key Conclusion

- **GATv2 runs on GraphSAGE's hyperparameters** (lr 0.01, 50 epochs, 2 layers, 64 dims). Deliberate - tuning would break the only-the-convolution-changes contract.
- **Are the added edges really long-range?** For every edge the role graph adds, we measured how many hops apart its two nodes are in the original graph (`experiments/edge_distance.py`, 6 graphs, K=10): median 3-6 hops, and only 14% are 1-2 hops apart. So they are not short links that attention on the original graph could already reach.


## Q5 · What kind of graph needs augmentation, and what kind does not?

Q2 found the two groups. This describes them. Not a rule and not a predictor - we are looking at the graphs we already
measured to see what the two cases have in common.

The 9 graphs where GATv2 alone lost against the 14 where it won. The 7 no-difference graphs are left out, so n = 23.
15 original-graph properties, reused from earlier modules' measurements.


In [6]:
prof = pd.read_csv("results/encoder_profile_groups.csv").query("label == 'needs'").set_index("property")
MEANING = {
    "degree_skew":          ("a few very large hubs", "degrees fairly even"),
    "degree_assortativity": ("hubs attach to leaves", "similar nodes attach to each other"),
    "avg_clustering":       ("few triangles, open neighbourhoods", "dense, closed neighbourhoods"),
    "degree_gini":          ("edges concentrated on few nodes", "edges shared out evenly"),
    "density":              ("sparse", "denser"),
}
show(pd.DataFrame([{"property": k.replace("_", " "),
                    "needs augmentation (9)": f"{prof.loc[k, 'augmentation_helps']:g}",
                    "does not (14)": f"{prof.loc[k, 'does_not']:g}",
                    "what that means": f"{a}  ->  {b}"} for k, (a, b) in MEANING.items()]))
print("properties that do NOT separate the two groups: homophily, neighbour predictability, number of classes, size")

property,needs augmentation (9),does not (14),what that means
degree skew,14.4123,2.3578,a few very large hubs -> degrees fairly even
degree assortativity,-0.1152,0.0273,hubs attach to leaves -> similar nodes attach to each other
avg clustering,0.2009,0.3599,"few triangles, open neighbourhoods -> dense, closed neighbourhoods"
degree gini,0.6273,0.483,edges concentrated on few nodes -> edges shared out evenly
density,0.0014,0.006,sparse -> denser


properties that do NOT separate the two groups: homophily, neighbour predictability, number of classes, size


In [7]:
# Summary: the same medians, read as high / low rather than as raw numbers.
SUMMARY = [
    ("degree_skew",          "hub dominance",              "HIGH", "LOW"),
    ("degree_gini",          "edges concentrated on a few nodes", "HIGH", "LOW"),
    ("degree_assortativity", "who connects to whom",       "NEGATIVE - hubs to leaves", "POSITIVE - like to like"),
    ("avg_clustering",       "triangles / local clustering", "LOW", "HIGH"),
    ("density",              "density",                    "LOW", "HIGH"),
    ("homophily_adjusted",   "homophily",                  "same", "same"),
    ("nodes",                "graph size",                 "no pattern", "no pattern"),
]
show(pd.DataFrame([{"what we measured": name,
                    "needs augmentation (9 graphs)": f"{hi}   ({prof.loc[k, 'augmentation_helps']:g})",
                    "does not need it (14 graphs)":  f"{lo}   ({prof.loc[k, 'does_not']:g})"}
                   for k, name, hi, lo in SUMMARY]))

what we measured,needs augmentation (9 graphs),does not need it (14 graphs)
hub dominance,HIGH (14.4123),LOW (2.3578)
edges concentrated on a few nodes,HIGH (0.6273),LOW (0.483)
who connects to whom,NEGATIVE - hubs to leaves (-0.1152),POSITIVE - like to like (0.0273)
triangles / local clustering,LOW (0.2009),HIGH (0.3599)
density,LOW (0.0014),HIGH (0.006)
homophily,same (0.0899),same (0.0949)
graph size,no pattern (9498),no pattern (7598)


**A5.** The two cases differ in **how the graph spreads its edges**, not in its labels.

- **Augmentation still needed** - sparse, a few dominant hubs, hubs wired to leaves, almost no triangles. The neighbours a node can reach on its own edges do not carry the signal, so extra long-range edges are what supply it.
- **Augmentation not needed** - denser and locally clustered, with degrees spread evenly. The neighbourhood already holds the information, so attention over the existing edges is enough.

This matches the mechanism: the edges the role graph adds sit a median 3-6 hops away, which a 2-layer GATv2 cannot see, and going deep enough to see them would oversmooth. Where the local neighbourhood is already informative that reach buys nothing; where it is not, it is the whole gain.

Homophily, neighbour predictability, class count and graph size show no separation at all (|rho| <= 0.19) - a useful negative, since homophily is what stage 1 keys on.
